# 0. 라이브러리 인스톨

In [ ]:
!pip install lomo-optim optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 30.8 MB/s eta 0:00:00


# 1. 사용 라이브러리 임포트

In [ ]:
import torch    # 딥러닝 학습을 위한 torch
import json     # 데이터를 불러올 json
import os
torch.manual_seed(123)  # 토치의 시드를 설정하여 같은 값으로 디버깅

In [ ]:
# device GPU(cuda) 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
# 데이터를 불러올 구글 드라이브 임포트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. 사용 모델 불러오기 - Gemma3 1B

In [ ]:
# 허깅페이스 공식 모델 불러오는 법
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

# Qwen3 1.7B 모델 불러오기
model_name = "google/gemma-3-1b-it"

# 모델의 가중치를 32비트가 아닌 16비트로 불러오기 -> GPU 메모리 사용량을 줄이고, 계산 속도도 빨라짐
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


# 토크나이저, 모델 불러오기
# 왼쪽 정렬을 하면 마지막 단어가 <pad>로 의미가 없음
# 그래서 오른쪽 정렬을 하면 항상 마지막 단어가 유의미해짐
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")  # 토크나이저 설정
# 모델 설정
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype
    )

model.to(device)        # 모델 GPU 올리기

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

# 3. 데이터 불러오기

In [ ]:
# 허깅페이스 개선 방법
from datasets import load_dataset

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

data_path = "/content/drive/MyDrive/main/Model_Train/data_set/train_data.jsonl"

# 데이터 셋 불러오기
dataset = load_dataset("json", data_files=data_path, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from datasets import load_dataset

# 1. 각 파일의 경로를 변수로 지정합니다.
train_file_path = "/content/drive/MyDrive/main/Model_Train/data_set/train_data.jsonl"
test_file_path = "/content/drive/MyDrive/main/Model_Train/data_set/test_data.jsonl"

# 2. data_files 인자에 딕셔너리 형태로 전달합니다.
#    'key'가 split의 이름이 되고, 'value'가 해당 파일의 경로가 됩니다.
data_files = {
    "train": train_file_path,
    "test": test_file_path
}

# 3. load_dataset 함수를 호출합니다.
#    이때 split 인자는 생략합니다.
all_datasets = load_dataset("json", data_files=data_files)

# --- 결과 확인 ---
print("전체 데이터셋 정보:")
print(all_datasets)

# 4. 각 데이터셋에 접근할 수 있습니다.
train_dataset = all_datasets["train"]
test_dataset = all_datasets["test"]

print(f"\n학습 데이터셋 크기: {len(train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

전체 데이터셋 정보:
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 16865
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1447
    })
})

학습 데이터셋 크기: 16865
테스트 데이터셋 크기: 1447


In [ ]:
# --- 전처리 함수 정의 ---
# 전체 대화 내용을 모델이 학습할 수 있는 단일 텍스트 시퀀스로 변환하는 함수
def formatting_prompts_func(examples):
    # 주어진 대화 내역을 질문과 답변으로 분리
    questions = examples["question"]
    answers = examples["answer"]
    texts = []

    for question, answer in zip(questions, answers):

        # Qwen3의 공식 채팅 템플릿을 사용하여 전체 대화 텍스트 생성
        messages = [
            {"role": "system", "content": "You are an assistant that explains terms about ship building."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]

        # add_generation_prompt=False: 학습 데이터에는 답변 생성 유도 프롬프트가 필요 없음
        # enable_thinking: 더 깊게 생각하기 모드, 기본설정 True
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        texts.append(text)

    return { "text": texts }    # 딕셔너리 형태로 return

In [ ]:
processed_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)
print("\n전처리된 데이터 예시:\n", processed_dataset[1]['text'])

Map:   0%|          | 0/16865 [00:00<?, ? examples/s]


전처리된 데이터 예시:
 <bos><start_of_turn>user
You are an assistant that explains terms about ship building.

According to the report, a problem occurred in the DECIBEL section.<end_of_turn>
<start_of_turn>model
보고서에 따르면, 데시벨(소음 측정단위) 부분에서 문제가 발생했습니다.<end_of_turn>



In [ ]:
test_data = test_dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)
print("\n전처리된 데이터 예시:\n", test_data[1]['text'])

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]


전처리된 데이터 예시:
 <bos><start_of_turn>user
You are an assistant that explains terms about ship building.

Tighten the penetration assembly according to the Gas Brazing specification.<end_of_turn>
<start_of_turn>model
관통부 시공은 가스 경납땜 규격에 맞춰 체결해 주세요.<end_of_turn>



In [ ]:
# --- ✨ 2. 누락된 토큰화 단계 추가 ✨ ---
def tokenize_function(examples):
    # 'text' 필드를 토큰화하여 'input_ids'와 'attention_mask'를 생성합니다.
    return tokenizer(
        examples["text"],
        truncation=True,      # max_length보다 길면 자르기
        max_length=2048,      # 모델이 처리할 최대 길이
    )

# 포맷팅된 데이터셋 전체에 토큰화 함수를 적용합니다.
# 이제 데이터셋에는 'input_ids'와 'attention_mask' 필드가 포함됩니다.
tokenized_train_dataset = processed_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # 더 이상 필요 없는 'text' 필드 제거
)

tokenized_test_dataset = test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # 더 이상 필요 없는 'text' 필드 제거
)

Map:   0%|          | 0/16865 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

# 4. Train, Test set 분리

In [ ]:
from torch.utils.data import random_split
from transformers import DataCollatorForLanguageModeling

total_size = len(processed_dataset)
train_size = 5000
test_size = 1000
unused_size = total_size - train_size - test_size
train_dataset, test_dataset, _ = random_split(tokenized_train_dataset, [train_size, test_size, unused_size])

# 테스트 데이터는 테스트 데이터로 덮어 씌우기
test_dataset = tokenized_test_dataset

print(f"전체 데이터셋 크기: {total_size}")
print(f"학습 데이터셋 크기: {len(train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

전체 데이터셋 크기: 16865
학습 데이터셋 크기: 5000
테스트 데이터셋 크기: 1447


In [ ]:
from torch.utils.data import DataLoader

# 허깅페이스의 데이터 콜렉터 객체를 생성해서 더욱 편하게 나눠줌 (자동 마스킹, 패딩, 저장 등)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# train, test 로더 설정
train_loader = DataLoader(
    train_dataset,
    batch_size=4, # 예시 배치 사이즈
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)

# ★ Optuna 하이퍼파라미터 조정

In [ ]:
import torch
import optuna
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_scheduler
from tqdm.auto import tqdm
from optuna.exceptions import TrialPruned

# LOMO 옵티마이저 import
from lomo_optim import   Lomo
from lomo_optim import AdaLomo

In [ ]:
# 2. Objective 함수 정의 (LOMO Full Fine-tuning 용)
def objective(trial):
    # =========================================
    # 파라미터 범주 설정
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    num_train_epochs = trial.suggest_int("num_train_epochs", 1, 3)

    # --- 메모리 관리를 위한 하이퍼파라미터 고정 ---
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8])

    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.05)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.1)
    lr_scheduler_type = trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine", "cosine_with_restarts"])

    # ===========================================
    # 매 시도 마다 모델, 데이터 로더, 옵티마이저 등 새로 초기화
    # 모델 설정
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=model_dtype)
    model.resize_token_embeddings(len(tokenizer))
    model.gradient_checkpointing_enable()

    model.to(device)

    # 2. 데이터 로더 생성
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=data_collator)

    # 3. ✨ LOMO 옵티마이저 설정 ✨
    optimizer = Lomo(model, lr=learning_rate, weight_decay=weight_decay)

    # 4. 학습률 스케줄러 설정
    num_training_steps = num_train_epochs * len(train_loader)
    num_warmup_steps = int(num_training_steps * warmup_ratio)
    lr_scheduler = get_scheduler(
        name=lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    # ===========================================
    # 학습 및 평가 루프
    for epoch in range(num_train_epochs):
        model.train()
        progress_bar = tqdm(train_loader, desc=f"Trial {trial.number} Epoch {epoch+1}/{num_train_epochs}", leave=False)

        for batch in progress_bar:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            # LOMO는 backward()에서 파라미터 업데이트까지 처리
            loss.backward()

            optimizer.zero_grad()
            lr_scheduler.step()
            progress_bar.set_postfix(loss=loss.item())

        # --- 프루닝(가지치기) 로직 ---
        model.eval()
        total_eval_loss = 0
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                total_eval_loss += loss.item()

        avg_eval_loss = total_eval_loss / len(test_loader)
        trial.report(avg_eval_loss, epoch)

        if trial.should_prune():
            raise TrialPruned()

    return avg_eval_loss

In [ ]:
# -------------------------------------------------------------------
# ## 3. Study 객체 생성 및 최적화 실행
# -------------------------------------------------------------------
from optuna.pruners import MedianPruner

study = optuna.create_study(direction="minimize", pruner=MedianPruner())
study.optimize(objective, n_trials=10) # 10번의 다른 조합으로 시도

best_params = study.best_params

# --- 결과 확인 ---
print("="*50)
print("최적화 종료!")
print("최고 점수 (loss):", study.best_trial.value)
print("최적 하이퍼파라미터:", study.best_params)
print("="*50)

[I 2025-10-17 13:11:43,831] A new study created in memory with name: no-name-cae14fcd-25ed-4d57-9842-ba3ec8da5fad


Trial 0 Epoch 1/2:   0%|          | 0/625 [00:00<?, ?it/s]

Trial 0 Epoch 2/2:   0%|          | 0/625 [00:00<?, ?it/s]

[I 2025-10-17 13:25:14,954] Trial 0 finished with value: 3.1713460624547296 and parameters: {'learning_rate': 3.211820720520105e-05, 'num_train_epochs': 2, 'batch_size': 8, 'weight_decay': 0.017741799273919175, 'warmup_ratio': 0.058242789211884465, 'lr_scheduler_type': 'cosine_with_restarts'}. Best is trial 0 with value: 3.1713460624547296.


Trial 1 Epoch 1/3:   0%|          | 0/625 [00:00<?, ?it/s]

...

Trial 9 Epoch 1/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Trial 9 Epoch 2/3:   0%|          | 0/1250 [00:00<?, ?it/s]

Trial 9 Epoch 3/3:   0%|          | 0/1250 [00:00<?, ?it/s]

[I 2025-10-17 17:38:11,842] Trial 9 finished with value: 3.306113318185121 and parameters: {'learning_rate': 2.197332053173538e-05, 'num_train_epochs': 3, 'batch_size': 4, 'weight_decay': 0.03298039326245661, 'warmup_ratio': 0.042086981984140626, 'lr_scheduler_type': 'cosine_with_restarts'}. Best is trial 8 with value: 3.1234273831488677.


최적화 종료!
최고 점수 (loss): 3.1234273831488677
최적 하이퍼파라미터: {'learning_rate': 3.936774351142264e-05, 'num_train_epochs': 1, 'batch_size': 8, 'weight_decay': 0.014013236653321805, 'warmup_ratio': 0.03824053083591647, 'lr_scheduler_type': 'cosine_with_restarts'}


In [ ]:
from optuna.pruners import MedianPruner

study = optuna.create_study(direction="minimize", pruner=MedianPruner())
study.optimize(objective, n_trials=1)

[I 2025-10-17 11:53:47,974] A new study created in memory with name: no-name-0327609c-e35c-430c-a911-ccd104c41075
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Trial 0 Epoch 1/3:   0%|          | 0/2500 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Trial 0 Epoch 2/3:   0%|          | 0/2500 [00:00<?, ?it/s]

Trial 0 Epoch 3/3:   0%|          | 0/2500 [00:00<?, ?it/s]

[I 2025-10-17 13:10:46,523] Trial 0 finished with value: 3.492224555977142 and parameters: {'learning_rate': 4.622532574041948e-05, 'num_train_epochs': 3, 'batch_size': 2, 'weight_decay': 0.04378885652680584, 'warmup_ratio': 0.047452654172959, 'lr_scheduler_type': 'cosine_with_restarts'}. Best is trial 0 with value: 3.492224555977142.


# 5. LOMO로 학습 진행

In [ ]:
from tqdm.auto import tqdm
from transformers import get_scheduler
from lomo_optim import AdaLomo


# --- 하이퍼 파라미터 설정 ---
learning_rate = 1e-4
num_epochs = 2
# -----------------------------

# LOMO는 AdamW와 같은 옵티마이저 상태를 저장하지 않아 메모리를 절약합니다.
optimizer = AdaLomo(model, lr=learning_rate)

# 학습률 스케줄러 설정
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0, # LOMO 사용 시에는 웜업을 사용하지 않는 경우가 많습니다.
    num_training_steps=num_training_steps
)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        # DataCollator가 반환한 배치를 GPU로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # 1. Forward Pass: 모델을 통해 예측(logits)과 손실(loss)을 계산
        outputs = model(**batch)
        loss = outputs.loss

        # 2. Backward Pass + Parameter Update (LOMO의 핵심)
        # LOMO는 backward() 호출 시 내부적으로 파라미터 업데이트까지 수행합니다.
        loss.backward()

        # 3. 그래디언트 초기화 및 스케줄러 스텝
        # backward() 후에 그래디언트를 초기화합니다.
        optimizer.zero_grad()
        lr_scheduler.step()

        progress_bar.set_postfix(loss=loss.item())

Epoch 1/2:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2/2:   0%|          | 0/1250 [00:00<?, ?it/s]

# 6. 성능 평가

In [ ]:
import math

model.eval()
total_eval_loss = 0

# 평가 시에는 가중치를 업데이트하지 않으므로, 불필요한 계산을 막아 메모리를 절약하고 속도를 높입니다.
with torch.no_grad():
    # test_loader를 사용하여 평가 데이터에 대한 루프 실행
    for batch in tqdm(test_loader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward Pass 실행
        outputs = model(**batch)
        loss = outputs.loss

        # 각 배치의 loss를 누적
        total_eval_loss += loss.item()

# 3. 평균 평가 손실(Average Evaluation Loss) 계산
avg_eval_loss = total_eval_loss / len(test_loader)

# 4. 퍼플렉시티(Perplexity) 계산
#    Perplexity는 e^(loss) 입니다. 값이 낮을수록 모델이 다음 단어를 잘 예측한다는 의미입니다.
try:
    perplexity = math.exp(avg_eval_loss)
except OverflowError:
    perplexity = float("inf") # loss가 너무 클 경우 무한대로 표시

# --- 5. 평가 결과 출력 ---
print("\n--- 평가 결과 ---")
print(f"평균 평가 손실 (Average Eval Loss): {avg_eval_loss:.4f}")
print(f"퍼플렉시티 (Perplexity): {perplexity:.4f}")
print("="*20)

Evaluating:   0%|          | 0/362 [00:00<?, ?it/s]


--- 평가 결과 ---
평균 평가 손실 (Average Eval Loss): 2.4700
퍼플렉시티 (Perplexity): 11.8227


# 7. 모델 저장

In [ ]:
# --- 최종 모델 저장 (Hugging Face 형식) ---

# 1. 저장할 '폴더'의 경로를 지정합니다. (파일 이름이 아님)
output_dir = "/content/drive/MyDrive/main/Model_Train/best_model/Qwen3_(1dot7B)_LOMO"

# 2. .save_pretrained() 메서드를 사용하여 모델과 토크나이저를 저장합니다.
#    이 메서드가 알아서 safetensors와 config.json 등을 생성합니다.
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ 최종 모델이 Hugging Face 형식으로 '{output_dir}' 폴더에 저장되었습니다.")

# (확인) 저장된 파일 목록을 출력해 봅니다.
# print("\n--- 저장된 파일 목록 ---")
# !ls -l {output_dir}


✅ 최종 모델이 Hugging Face 형식으로 '/content/drive/MyDrive/main/Model_Train/best_model/Qwen3_(1dot7B)_LOMO' 폴더에 저장되었습니다.


In [ ]:
from transformers import TextStreamer

# 모델을 평가 모드로 설정
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_answer(question):
    """
    올바른 채팅 템플릿을 사용하여 답변을 생성하는 함수.
    """
    # 1. 시스템 메시지와 사용자 질문으로 대화 형식 구성
    messages = [
        {"role": "system", "content": "You are an assistant that explains terms about ship building."},
        {"role": "user", "content": question}
    ]

    # 2. tokenizer.apply_chat_template을 사용하여 Qwen3의 공식 프롬프트 형식으로 변환
    #    add_generation_prompt=True가 모델에게 답변을 시작하라는 신호를 줍니다.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 3. 프롬프트를 토큰화하여 모델 입력으로 변환
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 4. 모델을 통해 답변 생성
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512)

    # 5. 생성된 결과에서 입력 프롬프트 부분을 제외하고 디코딩
    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    return answer

# --- 테스트 ---
while True:
    my_question = input()

    if my_question == '탈출':
        break

    final_answer = generate_answer(my_question)
    print(final_answer)

Draft의 뜻이 뭐야?
The term "Draft" in shipbuilding refers to the distance between the waterline and the lowest point of the ship.
What is mean about draft?
The draft refers to the depth of the hull below the waterline.


KeyboardInterrupt: Interrupted by user

# ★★ Optuna로 최적화 한 하이퍼파라미터로 다시 학습

In [ ]:
print(best_params)

{'learning_rate': 1.6813678226351355e-05, 'num_train_epochs': 1, 'batch_size': 2, 'weight_decay': 0.02894632801815435, 'warmup_ratio': 0.05999976203652793, 'lr_scheduler_type': 'cosine_with_restarts'}


In [ ]:
best_params = {'learning_rate': 1.6813678226351355e-05, 'num_train_epochs': 1, 'batch_size': 2, 'weight_decay': 0.02894632801815435, 'warmup_ratio': 0.05999976203652793, 'lr_scheduler_type': 'cosine_with_restarts'}

In [ ]:
# 최종 학습을 위한 모델 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype
    )

model.resize_token_embeddings(len(tokenizer)) # 어휘 크기 동기화
model.gradient_checkpointing_enable()         # 메모리 최적화
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], collate_fn=data_collator)

optimizer = Lomo(model, lr=best_params['learning_rate'])

num_epochs = best_params['num_train_epochs']
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(num_training_steps * best_params['warmup_ratio'])

lr_scheduler = get_scheduler(
    name=best_params['lr_scheduler_type'],
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
print("\n--- 최종 모델 학습 시작 ---")
for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Final Training Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.zero_grad()
        lr_scheduler.step()
        progress_bar.set_postfix(loss=loss.item())

print("✅ 최종 학습 완료!")


--- 최종 모델 학습 시작 ---


Final Training Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


✅ 최종 학습 완료!


In [ ]:
import math
# --- 4. 최종 성능 평가 ---
print("\n--- 최종 모델 성능 평가 시작 ---")
model.eval()
total_eval_loss = 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Final Evaluation"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_eval_loss += loss.item()

avg_eval_loss = total_eval_loss / len(test_loader)
perplexity = math.exp(avg_eval_loss)

print("\n--- 최종 평가 결과 ---")
print(f"평균 평가 손실 (Loss): {avg_eval_loss:.4f}")
print(f"퍼플렉시티 (Perplexity): {perplexity:.4f}")
print("="*25)


--- 최종 모델 성능 평가 시작 ---


Final Evaluation:   0%|          | 0/724 [00:00<?, ?it/s]


--- 최종 평가 결과 ---
평균 평가 손실 (Loss): 3.2584
퍼플렉시티 (Perplexity): 26.0075


In [ ]:
output_dir = "/content/drive/MyDrive/main/Model_Train/best_model/Gemma3_(1B)_LOMO_5000"

# 2. .save_pretrained() 메서드를 사용하여 모델과 토크나이저를 저장합니다.
#    이 메서드가 알아서 safetensors와 config.json 등을 생성합니다.
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ 최종 모델이 Hugging Face 형식으로 '{output_dir}' 폴더에 저장되었습니다.")


✅ 최종 모델이 Hugging Face 형식으로 '/content/drive/MyDrive/main/Model_Train/best_model/Gemma3_(1B)_LOMO_5000' 폴더에 저장되었습니다.
